# 03 · The Loss-Function Experiments

The experiment stage: the pre-registered hypotheses, the run matrix, the sweep and confirmation launches (RUN LATER), the registry-driven analysis, the single test pass, and the **pre-written interpretation branches** to be selected once results exist. Every training cell is GPU RUN LATER; the analysis cells run on CPU and will populate automatically once the committed CSVs exist.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (hypotheses), §3 (design), §7 (protocol), §15 (interpretation matrix).
- **Registry:** `../results/registry.csv`; eval CSVs: `../results/*_per_track.csv`.
- **Discipline:** the 50 test tracks are read **exactly once** (§7.4, gate G3), after the validation analysis below is frozen.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")


## 1 · Pre-registration recap (verbatim from MASTER_PLAN §2)

> **H-01a (primary).** Training the fixed U-Net on negative time-domain SI-SDR does **not** improve vocals SI-SDR at evaluation relative to L1-magnitude training.
> **Decision rule:** on the primary protocol (reduced budget, 3 seeds, 14-track validation, §7.2) let Δ = mean val SI-SDR(sisdr-arm) − mean val SI-SDR(l1-arm), and let σ_seed = the pooled between-seed std of the two arms.
> - **Supported** if Δ ≤ +σ_seed (SI-SDR training buys nothing beyond seed noise) **and** the full-budget test-set pair does not reverse this by more than the paired bootstrap 95 % CI.
> - **Refuted** if Δ > +σ_seed on validation **and** the full-budget test pair confirms (paired per-track delta > 0 with 95 % bootstrap CI excluding 0).
> - **Mixed** otherwise (val and test disagree, or seed band overlaps ambiguously) — reported as mixed, with variance discussion.

> **H-01b (secondary).** Adding a multi-resolution STFT auxiliary term (λ = 0.5) to L1 reduces **audible artifacts** at approximately equal SI-SDR.
> **Decision rule:** "approximately equal SI-SDR" = |SI-SDR(l1mrstft) − SI-SDR(l1)| ≤ σ_seed on validation. Given that, **supported** if the blind listening check (§7.5) prefers the MR-STFT arm on the artifact question in ≥ 60 % of clip-level pairwise judgments; **refuted** if ≤ 40 %; **inconclusive** between. If the SI-SDR-equality precondition fails, H-01b is scored **not evaluable as posed** and we report the quality/artifact tradeoff instead. (Pre-registered because the Bake-Off paper, arXiv 2507.06917, found BSS-Eval SDR is already the *best* perceptual proxy for vocals — so we deliberately do not expect MR-STFT to move SDR itself.)

## 2 · Run matrix (MASTER_PLAN §3.2)

| Stage | Runs | Budget (steps) | Seeds | Purpose |
|---|---|---|---|---|
| Sweep (primary evidence) | 5 arms × 3 seeds = **15** | REDUCED = 16,000 | {0,1,2} | H-01a on val; seed band |
| Confirmation | top-2 by mean val SI-SDR (+ l1mag) = **3** | FULL = 40,000 | {0} | ranking stability; test/listen checkpoints |
| Exploratory (budget-gated) | l1mrstft λ ∈ {0.25, 1.0} = **2** | REDUCED | {0} | λ sensitivity (labeled exploratory) |

**Budget-scaling rule (pre-registered):** if a FULL run projects to > 12 h on the available GPU, halve both budgets for *all* arms (`run_sweep.py --halve`); comparability is preserved because every arm shares the budget.

### 2.1 Launch the sweep
> ⚠️ **RUN THIS LATER** — 15-run sweep (5 arms × 3 seeds), REDUCED  ·  _≈ 20–27 T4-h · resumable_

Resumable: a Colab disconnect costs minutes (1k-step checkpoints + registry).

In [ ]:
# ⚠️ RUN THIS LATER (GPU). Iterates the 15 configs; skips completed runs.
# !python scripts/run_sweep.py --stage reduced
# (optional, budget-gated) exploratory λ pair:
# !python scripts/run_sweep.py --stage reduced --exploratory
print('Sweep launch is RUN LATER — needs a GPU and decoded data.')

## 3 · Registry-driven analysis (val)
These cells read `results/registry.csv` and render the ranking with the seed band. They are written to work the moment the sweep populates the registry; before then they print a friendly 'no runs yet'. **This is the H-01a decision surface** — all on validation, no test contact.

In [ ]:
import pandas as pd, numpy as np
from singnet.train import read_registry
reg = read_registry('01-loss-function-study/results/registry.csv')
reduced = reg[reg['budget'] == 16000] if len(reg) else reg
if len(reduced):
    table = (reduced.groupby('arm')['best_val_sisdr']
             .agg(['mean', 'std', 'count']).sort_values('mean', ascending=False))
    display(table)
else:
    print('no runs yet — this table fills once run_sweep.py has populated the registry.')

### 3.1 Ranking with the seed-noise band (H-01a)
*Figure to render:* a bar per arm = mean val SI-SDR, whisker = between-seed std, with a shaded pooled-σ_seed band around the `l1mag` bar. **How to read:** if the `sisdr` bar sits inside the `l1mag` band, Δ ≤ σ_seed → H-01a **supported** on validation; if it clears the band upward, that points toward **refuted** (pending the test pass).

In [ ]:
# Fills once the registry has ≥1 seed per arm. Draws the seed band from the 3-seed spread.
import matplotlib.pyplot as plt
if len(reduced):
    stats = reduced.groupby('arm')['best_val_sisdr'].agg(['mean', 'std'])
    ax = stats['mean'].plot.bar(yerr=stats['std'], capsize=4)
    ax.set_ylabel('val SI-SDR (dB)'); ax.set_title('Loss arms with seed-noise band')
    if 'l1mag' in stats.index:
        s = float(stats.loc['l1mag', 'std']); m = float(stats.loc['l1mag', 'mean'])
        ax.axhspan(m - s, m + s, alpha=0.15)   # pooled-σ_seed band around l1mag
else:
    print('no runs yet.')

### 3.2 Training curves & per-track val scatter
*Figures:* (a) val SI-SDR vs step per arm (every-2k evals) — convergence and stability; (b) per-track val SI-SDR scatter `sisdr` vs `l1mag` with the y=x line — shows *which* tracks move, not just the mean. RUN LATER once checkpoints/logs exist.

In [ ]:
# ⚠️ RUN THIS LATER — reads per-run val logs / checkpoints written by the sweep.
print('curves + per-track scatter render once the sweep has produced logs.')

### 3.3 Mask / spectrogram error galleries per arm
> ⚠️ **RUN THIS LATER** — predicted mask & error spectrograms per arm  ·  _~2 min · CPU per checkpoint_

*Figure:* for one validation chunk, each arm's predicted mask, the oracle IRM, and the magnitude-error spectrogram side by side. **How to read:** where an arm leaves residual accompaniment (bright error in dark-IRM regions) vs where it over-suppresses vocals — the qualitative signature behind the SI-SDR numbers and the H-01b artifact question.

In [ ]:
# ⚠️ RUN THIS LATER — needs the trained checkpoints. Uses singnet.eval + oracle_irm.
print('mask/error gallery renders from checkpoints (RUN LATER).')

## 4 · Confirmation + oracle/floor anchors
> ⚠️ **RUN THIS LATER** — 3 full-budget confirmation runs (top-2 + l1mag)  ·  _≈ 10–14 T4-h_

Run **after** the validation analysis above is frozen and committed (gate G3). `run_sweep.py --stage full` regenerates the 3 full configs from the frozen ranking and trains them.

In [ ]:
# ⚠️ RUN THIS LATER (GPU). Generates + runs the 3 FULL configs.
# !python scripts/run_sweep.py --stage full
print('Confirmation runs are RUN LATER.')

The **do-nothing floor** and **oracle IRM/IBM** lines (computed once, CPU) go on every results figure as headroom context (THEORY §2, §7.3).

## 5 · The single test pass (exactly once)
> ⚠️ **RUN THIS LATER** — score 3 checkpoints + floor + oracles on 50 test tracks  ·  _GPU minutes / CPU-ok_

**Gate G3 first:** registry complete for 18 runs; val analysis frozen & committed; test paths never appeared in any training config (grep-audited). Then run this **once**. It writes `results/test_per_track.csv` (SI-SDR primary) and, with `--museval`, `results/test_museval.csv` (BSS-Eval SDR — clearly-labelled secondary, never mixed with SI-SDR).

In [ ]:
# ⚠️ RUN THIS LATER — the ONE test pass for the whole direction (§7.4).
# !python scripts/evaluate.py --checkpoint <best.pt> --split test \
#     --shard-root $SHARD_ROOT --splits-csv 01-loss-function-study/configs/splits.csv \
#     --output-dir 01-loss-function-study/results --oracles --museval
print('Test pass is RUN LATER — run exactly once after G3.')

### 5.1 Paired statistics on the headline pairs
For (`sisdr` − `l1mag`) and (`l1mrstft` − `l1mag`): a **paired bootstrap 95 % CI** (10,000 resamples over the 50 tracks) and a **Wilcoxon signed-rank** two-sided p (THEORY §6). These are the only inferential statistics; everything else is descriptive.

In [ ]:
# Fills once results/test_per_track.csv exists.
from pathlib import Path
csv = Path('01-loss-function-study/results/test_per_track.csv')
if csv.exists():
    import pandas as pd, numpy as np
    from scipy.stats import wilcoxon
    df = pd.read_csv(csv).pivot_table(index='track', columns='system', values='sisdr_vocals')
    for a, b in [('singnet', 'do_nothing')]:   # replace with per-arm systems when available
        d = (df[a] - df[b]).dropna().values
        boot = [np.mean(np.random.choice(d, len(d), replace=True)) for _ in range(10000)]
        lo, hi = np.percentile(boot, [2.5, 97.5])
        w = wilcoxon(d)
        print(f'{a}-{b}: mean {d.mean():+.2f} dB, 95% CI [{lo:+.2f},{hi:+.2f}], Wilcoxon p={w.pvalue:.3g}')
else:
    print('no test CSV yet — run the test pass first.')

## 6 · Listening kit (H-01b evidence)
> ⚠️ **RUN THIS LATER** — render level-matched vocals + karaoke clips for 5 test tracks  ·  _~5 min · CPU_

Blind A/B: `l1` vs `l1mrstft` (primary) and `l1` vs `sisdr` (secondary), one question — *'which has fewer artifacts (musical noise, gurgling, phasiness)?'* — n ≥ 5 raters. Framed as a qualitative pilot, never statistical proof (§7.5).

In [ ]:
# ⚠️ RUN THIS LATER (CPU). Renders the blind-test clips.
# !python scripts/render_listening_kit.py --checkpoints <A> <B> <C> --labels l1 l1mrstft sisdr \
#     --shard-root $SHARD_ROOT --splits-csv 01-loss-function-study/configs/splits.csv
print('Listening-kit render is RUN LATER.')

**Result entry form (fill during the session):**

| clip | pair | preferred arm | quality slider (0–100) | rater notes |
|---|---|---|---|---|
| 1 | l1 vs l1mrstft |  |  |  |
| 2 | l1 vs l1mrstft |  |  |  |
| … | … |  |  |  |

Aggregate as preference counts + per-clip majorities; report rater agreement.

## 7 · Interpretation — pre-written branches (select one when results exist)

_These are pre-registered readings (MASTER_PLAN §15). They are written now, before any run, so the conclusion cannot be reverse-engineered from the data — **pick the branch the results select; do not edit the others.**_

### H-01a branches
- **[branch: H-01a SUPPORTED]** — Δ ≤ +σ_seed on val and the test pair does not reverse it. *Reading:* at compact scale, training on the eval metric buys nothing beyond seed noise — consistent with Gusó's verified scope and Demucs practice. *Consequence:* **keep `l1mag` as the project default loss** for directions 02–10.
- **[branch: H-01a REFUTED]** — Δ > +σ_seed on val and the test pair confirms (95 % CI excludes 0). *Reading:* direct metric optimization helps at small capacity; the literature tension is scale-dependent. *Consequence:* **adopt `sisdr` (with the −60 dBFS guard) as default** and flag re-run implications for later directions.
- **[branch: H-01a MIXED]** — val and test disagree, or the seed band overlaps ambiguously. *Reading:* the effect is within run-to-run instability. *Consequence:* **keep `l1mag`** (simplicity prior) and report the variance honestly.

### H-01b branches
- **[branch: H-01b SUPPORTED]** — SI-SDR-equal precondition holds and the listening check prefers MR-STFT ≥ 60 %. *Reading:* real artifact gains at equal SDR — the metric blind spot the Bake-Off predicts. *Consequence:* **adopt the MR-STFT auxiliary for shipped-model training.**
- **[branch: H-01b REFUTED / INCONCLUSIVE]** — preference ≤ 40 %, or between 40–60 %. *Reading:* artifact differences inaudible at this scale or swamped by rater noise. *Consequence:* **drop the aux term** (simplicity); document.
- **[branch: H-01b NOT EVALUABLE AS POSED]** — the SI-SDR-equality precondition fails. *Reading:* MR-STFT changed SDR, so the equal-SDR artifact question can't be asked cleanly. *Consequence:* report the quality/artifact tradeoff instead.
- **[branch: sisdr SKIP RATE > 20 %]** — *Reading:* SI-SDR loss is fragile on real music (silence-dominated chunks). *Consequence:* a standalone finding that **feeds Direction 08's silence framing.**

## 8 · Conclusions & what this changes for directions 02–10
- The winning-branch verdict sets the **default training loss** every later direction inherits (02 augmentation, 03 band-split, 05 LoRA, 06 robust, 08 silence, 10 distillation all train *some* loss).
- The measured **σ_seed** becomes the noise band those directions draw on their own sweep figures (project house style).
- The **`sisdr` skip-rate** and the silent-region EDA feed Direction 08.
- The shared, tested `singnet/` infrastructure (data, model, losses, train, eval) is reused verbatim — this direction pays that cost once.

*Fill the one-paragraph verdict here after selecting the branches above; it becomes the abstract line in `paper/PAPER.md`.*